# Base Year Data Engineering
**Purpose:** Build the parcel-level attributed dataset and TAZ-level input files
for the 2026 Housing EIS Travel Demand Model base year.

**Stages:**
1. Data Acquisition (web services)
2. Parcel Classification (VHR flag, TAU type, unit changes, spatial joins)
3. Occupancy Rate Application
4. IDW Interpolation
5. Census / Socioeconomic Attribution
6. Campground Processing
7. School Enrollment Processing
8. TAZ Aggregation & Output CSV Export
9. QA & Basin Summary

**Checkpoints** are saved as Parquet (no pickle version lock-in).

## Configuration
All constants live here. Update this cell when running for a new base year.

In [ ]:
from pathlib import Path

# Year filters
PARCEL_YEAR          = 2022
SCHOOL_YEAR          = "2021-2022"
CENSUS_YEAR          = 2022
# Occupancy rate timeframes to average (ISO date strings from the feature service)
OCCUPANCY_TIMEFRAMES = ["2022-06-01", "2022-08-01", "2022-09-01"]

# Fallback occupancy rates (used only when IDW also fails) 
DEFAULT_TAU_OCCUPANCY = 0.592337
DEFAULT_VHR_OCCUPANCY = 0.422337

# Population adjustment 
# ACS total vs model-calculated total from prior base year run.
ACS_TOTAL_POPULATION        = 53953
MODEL_CALCULATED_POPULATION = 52788.25363
HOUSEHOLD_SIZE_ADJUSTMENT   = ACS_TOTAL_POPULATION / MODEL_CALCULATED_POPULATION

# Paths 
SCRIPTS_DIR = Path().absolute()                    # .../Base/scripts
DATA_DIR    = SCRIPTS_DIR.parents[0] / "data" / "raw_data"
OUT_DIR     = SCRIPTS_DIR.parents[0] / "data" / "processed_data"
LOOKUP_DIR  = SCRIPTS_DIR / "Lookup_Lists"
WORKSPACE   = Path(r"C:\GIS\Scratch.gdb")        # ArcGIS scratch workspace

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint paths (Parquet files for intermediate outputs, used for debugging and to speed up development by avoiding re-running expensive steps) 
CKPT_SPATIAL_JOINS   = OUT_DIR / "parcel_spatial_joins.parquet"
CKPT_OCC_RATES       = OUT_DIR / "parcel_occupancy_rates.parquet"
CKPT_OCC_INTERP      = OUT_DIR / "parcel_occupancy_interpolated.parquet"
CKPT_SOCIOECONOMIC   = OUT_DIR / "parcel_socioeconomic.parquet"
CKPT_CAMPGROUND      = OUT_DIR / "campground_taz.parquet"
CKPT_SCHOOL          = OUT_DIR / "school_enrollment.parquet"
CKPT_OCC_RATES_TABLE = OUT_DIR / "occupancy_rates_table.parquet"
CKPT_PARCEL_FINAL    = OUT_DIR / "parcels_base_year.gdb.zip"

# Web service URLs 
BASE_URL = "https://maps.trpa.org/server/rest/services"
# REST endpoints for each layer used in the spatial joins and occupancy rate interpolation.
PARCEL_URL         = f"{BASE_URL}/Existing_Development/MapServer/2"
VHR_URL            = f"{BASE_URL}/VHR/MapServer/0"
TAZ_URL            = f"{BASE_URL}/Transportation_Planning/MapServer/6"
BLOCK_GROUPS_URL   = f"{BASE_URL}/Demographics/MapServer/27"
CENSUS_URL         = f"{BASE_URL}/Demographics/MapServer/28"
CAMPGROUND_URL     = f"{BASE_URL}/Recreation/MapServer/1"
CAMPVISITS_URL     = f"{BASE_URL}/Transportation_Planning/MapServer/14"
OCC_ZONES_URL      = f"{BASE_URL}/Transportation_Planning/MapServer/15"
OCC_RATES_URL      = f"{BASE_URL}/Transportation_Planning/MapServer/13"
SCHOOL_TABLE_URL   = f"{BASE_URL}/Transportation_Planning/MapServer/17"
SCHOOL_SPATIAL_URL = f"{BASE_URL}/Transportation_Planning/MapServer/16"

# Final parcel schema (no SHAPE -- geometry is not stored in parquet; re-attached where needed)
FINAL_SCHEMA = [
    "APN", "Residential_Units", "TouristAccommodation_Units", "CommercialFloorArea_SqFt",
    "RoomsRented_PerDay", "VHR_Occupancy_Rate", "TAU_Occupancy_Rate",
    "PrimaryResidence_Rate", "SecondaryResidence_Rate",
    "HighIncome_Rate", "MediumIncome_Rate", "LowIncome_Rate",
    "PersonsPerUnit", "TAU_TYPE", "VHR", "BLOCK_GROUP", "TAZ",
    "OCCUPANCY_ZONE", "JURISDICTION", "COUNTY", "OWNERSHIP_TYPE",
    "EXISTING_LANDUSE", "WITHIN_TRPA_BNDY", "WITHIN_BONUSUNIT_BNDY", "ZONING_ID", "PARCEL_ACRES", "PARCEL_SQFT"
]

print("Config loaded.")
print(f"  Data dir  : {DATA_DIR}")
print(f"  Output dir: {OUT_DIR}")

## Imports

In [ ]:
import pandas as pd
import numpy as np
import os

# ArcGIS / spatial
import arcpy
from arcgis.features import FeatureLayer, GeoAccessor, GeoSeriesAccessor

# Local utilities
from utils import (
    get_fs_data, get_fs_data_query,
    get_fs_data_spatial, get_fs_data_spatial_query,
    # renamecolumns, update_if_contains,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

arcpy.env.workspace = str(WORKSPACE)
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference("NAD 1983 UTM Zone 10N")
arcpy.env.overwriteOutput = True

print("Imports OK.")

## Target Output Schemas
Read the example input files from `Example_Input_File_Structure` to display
the exact column names, order, and dtypes that Stage 8 must produce.

In [ ]:
# â”€â”€ Target Output Schemas â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Read the reference CSVs from the model's Example_Input_File_Structure folder
# so we can verify our Stage 8 outputs match exactly.
EXAMPLE_DIR = Path(r"F:\Transportation\TravelDemandModel\Example_Input_File_Structure")

TARGET_FILES = [
    "OvernightVisitorZonalData_Summer.csv",
    "SchoolEnrollment.csv",
    "SocioEcon_Summer.csv",
    "VisitorOccupancyRates_Summer.csv",
]

target_schemas = {}
example_dfs    = {}
for fname in TARGET_FILES:
    df_ex = pd.read_csv(EXAMPLE_DIR / fname)
    example_dfs[fname] = df_ex
    target_schemas[fname] = {"columns": list(df_ex.columns), "dtypes": df_ex.dtypes.to_dict()}
    print(f"--- {fname} ---")
    print(f"  Columns ({len(df_ex.columns)}): {list(df_ex.columns)}")
    print(f"  Dtypes:")
    for col, dtype in df_ex.dtypes.items():
        print(f"    {col:40s} {dtype}")
    print(f"  Sample row: {df_ex.iloc[0].to_dict()}")
    print()

print("Target schemas and DataFrames loaded.")
print("  example_dfs keys:", list(example_dfs.keys()))


## Helper Functions

In [ ]:
def _to_arcpy_geom(geom, sr):
    """Convert arcgis API Geometry to arcpy Geometry for InsertCursor compatibility."""
    if geom is None:
        return None
    if hasattr(geom, "as_arcpy"):
        return geom.as_arcpy
    return arcpy.FromWKT(geom.WKT, sr)


def _sdf_to_fc(sdf, fc_path, fields, shape_col):
    """Write SDF fields to a real feature class via arcpy InsertCursor (avoids arcgis API bugs)."""
    import os
    sdf_clean = sdf[fields + [shape_col]].dropna(subset=[shape_col]).copy()
    if sdf_clean.empty:
        raise ValueError(f"No valid geometries in SDF for fields {fields}")

    first_geom = sdf_clean[shape_col].iloc[0]
    geom_map   = {"point": "POINT", "multipoint": "MULTIPOINT",
                  "polygon": "POLYGON", "polyline": "POLYLINE"}
    geom_type  = geom_map.get(getattr(first_geom, "type", "point").lower(), "POINT")

    # spatialReference from arcgis API geometries is a dict {wkid:...}, not arcpy.SpatialReference
    sr_raw = getattr(first_geom, "spatialReference", None)
    if isinstance(sr_raw, arcpy.SpatialReference):
        sr = sr_raw
    elif isinstance(sr_raw, dict):
        wkid = sr_raw.get("wkid") or sr_raw.get("latestWkid")
        sr = arcpy.SpatialReference(int(wkid)) if wkid else arcpy.SpatialReference(4326)
    else:
        sr = arcpy.SpatialReference(4326)

    ws, name = os.path.split(fc_path)
    arcpy.management.CreateFeatureclass(ws, name, geom_type, spatial_reference=sr)

    dtype_map = {"int64": "LONG", "float64": "DOUBLE"}
    for fld in fields:
        ftype = dtype_map.get(str(sdf_clean[fld].dtype), "TEXT")
        if ftype == "TEXT":
            arcpy.management.AddField(fc_path, fld, "TEXT", field_length=255)
        else:
            arcpy.management.AddField(fc_path, fld, ftype)

    with arcpy.da.InsertCursor(fc_path, fields + ["SHAPE@"]) as cur:
        for _, row in sdf_clean.iterrows():
            arcpy_geom = _to_arcpy_geom(row[shape_col], sr)
            cur.insertRow([row[f] for f in fields] + [arcpy_geom])


def arcpy_spatial_join_attr(target_sdf, join_sdf, join_field,
                             match_option="HAVE_THEIR_CENTER_IN", key_field="APN"):
    """Spatial join using real feature classes in scratchGDB; returns dict {key_field: join_field value}.
    Uses unique FC names (uuid) per call to avoid schema-lock conflicts from prior failed runs."""
    import os, uuid
    uid       = uuid.uuid4().hex[:8]
    scratch   = arcpy.env.scratchGDB
    target_fc = os.path.join(scratch, f"target_sj_{uid}")
    join_fc   = os.path.join(scratch, f"join_sj_{uid}")
    out_fc    = os.path.join(scratch, f"out_sj_{uid}")

    t_shape = next((c for c in target_sdf.columns if c.upper() == "SHAPE"), None)
    j_shape = next((c for c in join_sdf.columns  if c.upper() == "SHAPE"), None)
    if t_shape is None:
        raise KeyError(f"No SHAPE column in target SDF. Columns: {list(target_sdf.columns)}")
    if j_shape is None:
        raise KeyError(f"No SHAPE column in join SDF. Columns: {list(join_sdf.columns)}")

    try:
        _sdf_to_fc(target_sdf, target_fc, [key_field],  t_shape)
        _sdf_to_fc(join_sdf,   join_fc,   [join_field], j_shape)

        arcpy.analysis.SpatialJoin(
            target_features=target_fc, join_features=join_fc, out_feature_class=out_fc,
            join_operation="JOIN_ONE_TO_ONE", join_type="KEEP_ALL",
            match_option=match_option
        )

        # Read result with cursor -- avoids arcgis from_featureclass bugs
        fc_fields = [f.name for f in arcpy.ListFields(out_fc)]
        fc_upper  = {f.upper(): f for f in fc_fields}
        kf_actual = fc_upper.get(key_field.upper())
        jf_actual = fc_upper.get(join_field.upper())
        if kf_actual is None or jf_actual is None:
            raise KeyError(
                f"Column not found in spatial join output. "
                f"Looking for: {key_field} (found={kf_actual is not None}), "
                f"{join_field} (found={jf_actual is not None}). "
                f"Available: {fc_fields}"
            )
        result = {}
        with arcpy.da.SearchCursor(out_fc, [kf_actual, jf_actual]) as cur:
            for row in cur:
                if row[1] is not None:
                    result[row[0]] = row[1]
        return result

    finally:
        # Always clean up scratch FCs
        for fc in [out_fc, target_fc, join_fc]:
            if arcpy.Exists(fc):
                arcpy.management.Delete(fc)


def map_rates(df, rate_df, zone_col, rate_col, target_col):
    """Map a rate from rate_df (keyed by zone_col) onto df, writing to target_col."""
    rate_map = rate_df.set_index(zone_col)[rate_col].to_dict()
    df[target_col] = df[zone_col].map(rate_map)
    return df


def check_dupes(df, col, drop=True):
    """Report and optionally remove duplicate rows by col."""
    dupes = df[df.duplicated(subset=col, keep=False)]
    if len(dupes):
        print(f"  WARNING: {len(dupes)} duplicate rows on {col}")
        if drop:
            df = df.drop_duplicates(subset=col, keep="first")
            print(f"  Dropped. Remaining: {len(df)}")
    else:
        print(f"  No duplicates on {col}.")
    return df

def qa_summary(df, label, unit_cols=None):
    '''Print a concise QA snapshot: row count and column sums for unit_cols.'''
    print(f'=== QA: {label} ===')
    print(f'  Rows: {len(df):,}')
    if unit_cols:
        for col in unit_cols:
            if col in df.columns:
                print(f'  {col}: {df[col].fillna(0).sum():,.0f}')
            else:
                print(f'  {col}: (column not found)')


---
## Stage 1 - Get Data
Fetch all source datasets from TRPA web services.

**Outputs:** Raw DataFrames in memory for downstream stages

> If a service URL returns an error, correct the URL in Config and re-run this cell.

In [ ]:
print("Fetching parcels...")
sdfParcel_raw = get_fs_data_spatial_query(PARCEL_URL, query_params=f"Year = {PARCEL_YEAR}")
print(f"  Parcels        : {len(sdfParcel_raw):,}")

print("Fetching VHR parcel list...")
df_vhr_list = get_fs_data(VHR_URL)
print(f"  VHR records    : {len(df_vhr_list):,}")

print("Fetching TAZ...")
sdf_taz = get_fs_data_spatial(TAZ_URL)
print(f"  TAZ zones      : {len(sdf_taz):,}")

print("Fetching block groups...")
sdf_block_groups = get_fs_data_spatial(BLOCK_GROUPS_URL)
print(f"  Block groups   : {len(sdf_block_groups):,}")
print(f"  BG columns     : {list(sdf_block_groups.columns)}")
print(f"  TRPAID sample  : {sdf_block_groups['TRPAID'].head(3).tolist()}")

# PATCHED OUT -- Stage 5 now reads census block-group rates from the frozen
# CSV (census_block_group_rates_2022.csv) rather than this live ACS service.
# The service schema changed from wide to long format and the year_sample field
# was removed, making this fetch return mixed-year data (see Gotcha #11).
# Keep this block to document the original intended approach; revert by
# uncommenting these lines and deleting the patch cells in Stage 5.
#
# print("Fetching census demographics...")
# df_census = get_fs_data_query(CENSUS_URL, query_params=f"year_sample = {CENSUS_YEAR}")
# print(f"  Census records : {len(df_census):,}")

print("Fetching occupancy zones...")
sdf_occ_zones = get_fs_data_spatial(OCC_ZONES_URL)
print(f"  Occ zones      : {len(sdf_occ_zones):,}")

print("Fetching occupancy rates...")
# Occupancy rates are read from the raw CSV in Stage 3 (not from the web service).
# The web service schema changed and only returns monthly data for a subset of zones.

print("Fetching campgrounds...")
sdf_campgrounds = get_fs_data_spatial_query(CAMPGROUND_URL, query_params="RECREATION_TYPE='Campground'")
df_campvisits   = get_fs_data_query(CAMPVISITS_URL, query_params=f"Year = {PARCEL_YEAR}")
print(f"  Campgrounds    : {len(sdf_campgrounds):,}")

print("Fetching schools...")
df_school_table = get_fs_data_query(SCHOOL_TABLE_URL, query_params=f"Year = '{SCHOOL_YEAR}'")
sdf_schools     = get_fs_data_spatial(SCHOOL_SPATIAL_URL)
print(f"  School records : {len(df_school_table):,}")

print("\nAll services fetched.")

In [ ]:
# PATCHED OUT -- diagnostic cell used to identify the ACS service schema
# change described in Gotchas #9 and #11. The service now returns long-format
# data (variable_code/value rows) without a year_sample filter field.
# Leaving this active during Run All would add ~2 duplicate service fetches
# and print schema information that no longer matches the 2022 processing
# logic. Uncomment to re-inspect the live service schema if needed.
#
# # --- Quick diagnostic: inspect block groups and census field schemas ---
# import pandas as pd

# _bg   = get_fs_data_spatial(BLOCK_GROUPS_URL)
# _cen  = get_fs_data_query(CENSUS_URL, query_params=f"year_sample = {CENSUS_YEAR}")

# print("=== Block Groups (MapServer/27) ===")
# print(f"  Rows    : {len(_bg)}")
# print(f"  Columns : {[c for c in _bg.columns if c != 'SHAPE']}")
# print(f"  TRPAID dtype : {_bg['TRPAID'].dtype}")
# print(f"  TRPAID sample: {_bg['TRPAID'].head(5).tolist()}")

# print("=== Census Demographics (MapServer/28) ===")
# print(f"  Rows    : {len(_cen)}")
# print(f"  Columns : {list(_cen.columns)}")
# print(f"  TRPAID dtype : {_cen['TRPAID'].dtype}")
# _bg_cen = _cen[_cen['sample_level'] == 'block group'] if 'sample_level' in _cen.columns else _cen
# print(f"  TRPAID sample (BG level): {_bg_cen['TRPAID'].head(5).tolist()}")

# # Show any column in _bg that might match census TRPAID format
# print("=== Potential join-key columns in block groups layer ===")
# for col in [c for c in _bg.columns if c != 'SHAPE']:
#     sample = _bg[col].head(3).tolist()
#     print(f"  {col:30s}: {sample}")


---
## Stage 2 - Parcel Classification
Apply unit changes, assign VHR flag and TAU type, zero closed parcels,
then spatial-join to TAZ / block group / occupancy zone.

**Inputs:** `sdfParcel_raw`, `df_vhr_list`, `sdf_taz`, `sdf_block_groups`, `sdf_occ_zones`
**Checkpoint:** `parcel_spatial_joins.parquet`

### 2a - Apply known unit changes (development_2022_2025)

In [ ]:
df_dev = (
    pd.read_csv(LOOKUP_DIR / "development_2022_2025.csv")
    [["Fixed_APN", "Change"]]
    .rename(columns={"Fixed_APN": "APN"})
    .dropna(subset=["Change"])
    .astype({"Change": int})
)

sdfParcel = sdfParcel_raw.copy()
sdfParcel = sdfParcel.merge(df_dev, on="APN", how="left")
sdfParcel["Change"] = sdfParcel["Change"].fillna(0).astype(int)
sdfParcel["Residential_Units_Adjusted"] = (
    sdfParcel["Residential_Units"].fillna(0).astype(int) + sdfParcel["Change"]
).clip(lower=0)

changed = (sdfParcel["Change"] != 0).sum()
print(f"Unit changes applied to {changed} parcels")
print(f"Residential_Units_Adjusted total: {sdfParcel['Residential_Units_Adjusted'].sum():,}")

### 2b - VHR flag

In [ ]:
vhr_apns = set(df_vhr_list["APN"].dropna().astype(str))
sdfParcel["VHR"] = sdfParcel["APN"].astype(str).isin(vhr_apns).map({True: "Yes", False: "No"})
print(f"VHR parcels : {(sdfParcel['VHR'] == 'Yes').sum():,}")
print(f"Non-VHR     : {(sdfParcel['VHR'] == 'No').sum():,}")

### 2c - TAU type classification

In [ ]:
df_tau_lookup = pd.read_csv(LOOKUP_DIR / "lookup_tau_type.csv")[["APN", "TAU_Type"]]
tau_type_map  = df_tau_lookup.set_index("APN")["TAU_Type"].to_dict()

# Default: no TAU units -> N/A; has TAU units -> HotelMotel; lookup overrides with Casino/Resort
sdfParcel["TAU_TYPE"] = "N/A"
has_tau = sdfParcel["TouristAccommodation_Units"].fillna(0) > 0
sdfParcel.loc[has_tau, "TAU_TYPE"] = "HotelMotel"
sdfParcel["TAU_TYPE"] = sdfParcel["APN"].map(tau_type_map).fillna(sdfParcel["TAU_TYPE"])

print("TAU_TYPE distribution:")
print(sdfParcel["TAU_TYPE"].value_counts().to_string())

### 2d - Zero out closed tourist parcels

In [ ]:
df_closed   = pd.read_csv(LOOKUP_DIR / "closed_tourist_parcels.csv")
closed_apns = set(df_closed["APN"].astype(str))
n_closed    = sdfParcel["APN"].astype(str).isin(closed_apns).sum()
sdfParcel.loc[sdfParcel["APN"].astype(str).isin(closed_apns), "TouristAccommodation_Units"] = 0
print(f"Closed tourist parcels zeroed: {n_closed}")

### 2e - Spatial joins: TAZ, block group, occupancy zone

In [ ]:
# Spatial joins: TAZ, block groups, occupancy zones
print("Joining TAZ...")
sdfParcel["TAZ"] = sdfParcel["APN"].map(
    arcpy_spatial_join_attr(sdfParcel, sdf_taz, "TAZ")
)

# MapServer/27 contains mixed vintages (2000/2010/2020) and geography levels
# (Tract, Block Group). Filter to 2020 block-group records whose 16-char TRPAID
# matches the census TRPAID format directly 
sdf_bg_2020 = sdf_block_groups[
    (sdf_block_groups["YEAR"] == 2020) &
    (sdf_block_groups["TRPAID"].str.len() == 16)
].copy()
print(f"Block groups (2020 BG level): {len(sdf_bg_2020)} of {len(sdf_block_groups)} total")

print("Joining block groups...")
sdfParcel["BLOCK_GROUP"] = sdfParcel["APN"].map(
    arcpy_spatial_join_attr(sdfParcel, sdf_bg_2020, "TRPAID")
)

print("Joining occupancy zones...")
sdfParcel["OCCUPANCY_ZONE"] = sdfParcel["APN"].map(
    arcpy_spatial_join_attr(sdfParcel, sdf_occ_zones, "OccupancyRate_ZoneID")
)

# CSLT VHR parcels use a combined zone for occupancy rate averaging
cslt_vhr = (sdfParcel["JURISDICTION"] == "CSLT") & (sdfParcel["VHR"] == "Yes")
sdfParcel.loc[cslt_vhr, "OCCUPANCY_ZONE"] = "CSLT_ALL"
print(f"CSLT_ALL override applied to {cslt_vhr.sum()} parcels")


### 2f - Trim to final schema and save checkpoint

In [ ]:
for field in FINAL_SCHEMA:
    if field not in sdfParcel.columns:
        sdfParcel[field] = np.nan

keep_cols = FINAL_SCHEMA + ["Residential_Units_Adjusted", "Change"]
sdfParcel = sdfParcel[[c for c in keep_cols if c in sdfParcel.columns]].copy()

sdfParcel.to_parquet(CKPT_SPATIAL_JOINS, index=False)
print(f"Checkpoint saved: {CKPT_SPATIAL_JOINS}")

### QA - Stage 2

In [ ]:
qa_summary(sdfParcel, "After Spatial Joins",
           unit_cols=["Residential_Units", "Residential_Units_Adjusted", "TouristAccommodation_Units"])
print(f"  TAZ nulls        : {sdfParcel['TAZ'].isna().sum()}")
print(f"  BLOCK_GROUP nulls: {sdfParcel['BLOCK_GROUP'].isna().sum()}")
print(f"  OCC_ZONE nulls   : {sdfParcel['OCCUPANCY_ZONE'].isna().sum()}")
sdfParcel[["APN","TAZ","BLOCK_GROUP","OCCUPANCY_ZONE","VHR","TAU_TYPE"]].head()

---
## Stage 3 - Occupancy Rate Application
Average zone-level rates across configured timeframes, then map onto parcels.

**Inputs:** `parcel_spatial_joins.parquet`, `df_occ_rates_raw`
**Checkpoints:** `occupancy_rates_table.parquet`, `parcel_occupancy_rates.parquet`

In [ ]:
sdfParcel = pd.read_parquet(CKPT_SPATIAL_JOINS)
print(f"Loaded {len(sdfParcel):,} parcels from checkpoint")

In [ ]:
# Load pre-aggregated occupancy rates from raw data CSV.
# The live service (OCC_RATES_URL) has a different schema (ZoneID, Type, MonthNum)
# and only returns monthly rows for a subset of zones, causing most parcels to miss
# a zone match and fall back to the basin-wide default. The CSV already contains
# the correct per-zone, per-room-type rates averaged over the 2022 summer season,
# matching the source used in the 2022 RTP base year run.
df_occ_src = pd.read_csv(DATA_DIR / "occupancy_rates.csv")
print(f"Occupancy rate rows loaded: {len(df_occ_src)}")
print(f"Zones         : {sorted(df_occ_src['Zone_ID'].unique())}")
print(f"RoomTypes     : {sorted(df_occ_src['RoomType'].unique())}")

# TAU = hotel/casino/resort types; VHR = vacation home rentals
TAU_TYPES = ["HotelMotel", "Casino", "Resort"]

df_tau = (
    df_occ_src[df_occ_src["RoomType"].isin(TAU_TYPES)]
    .groupby("Zone_ID", as_index=False)["TRPA_OccRate"].mean()
    .rename(columns={"Zone_ID": "OCCUPANCY_ZONE", "TRPA_OccRate": "TAU_Occupancy_Rate"})
)
df_vhr = (
    df_occ_src[df_occ_src["RoomType"] == "VHR"]
    .groupby("Zone_ID", as_index=False)["TRPA_OccRate"].mean()
    .rename(columns={"Zone_ID": "OCCUPANCY_ZONE", "TRPA_OccRate": "VHR_Occupancy_Rate"})
)

# Merge TAU and VHR averaged rates per zone
df_occ_avg = df_tau.merge(df_vhr, on="OCCUPANCY_ZONE", how="outer")
df_occ_avg.to_parquet(CKPT_OCC_RATES_TABLE, index=False)
print(f"Zones with averaged rates: {len(df_occ_avg)}")
df_occ_avg


In [ ]:
# Map averaged zone rates onto parcels
sdfParcel = map_rates(sdfParcel, df_occ_avg, "OCCUPANCY_ZONE", "TAU_Occupancy_Rate", "TAU_Occupancy_Rate")
sdfParcel = map_rates(sdfParcel, df_occ_avg, "OCCUPANCY_ZONE", "VHR_Occupancy_Rate", "VHR_Occupancy_Rate")

# Zero rates where there are no relevant units
sdfParcel.loc[sdfParcel["TouristAccommodation_Units"].fillna(0) == 0, "TAU_Occupancy_Rate"] = 0
sdfParcel.loc[sdfParcel["VHR"] == "No",                               "VHR_Occupancy_Rate"] = 0

sdfParcel.to_parquet(CKPT_OCC_RATES, index=False)
print(f"Checkpoint saved: {CKPT_OCC_RATES}")

### QA - Stage 3

In [ ]:
tau_p = sdfParcel[sdfParcel["TouristAccommodation_Units"].fillna(0) > 0]
vhr_p = sdfParcel[sdfParcel["VHR"] == "Yes"]
print(f"TAU parcels missing rate: {tau_p['TAU_Occupancy_Rate'].isna().sum()}")
print(f"VHR parcels missing rate: {vhr_p['VHR_Occupancy_Rate'].isna().sum()}")
print(f"TAU rate range: {sdfParcel['TAU_Occupancy_Rate'].min():.3f} to {sdfParcel['TAU_Occupancy_Rate'].max():.3f}")
print(f"VHR rate range: {sdfParcel['VHR_Occupancy_Rate'].min():.3f} to {sdfParcel['VHR_Occupancy_Rate'].max():.3f}")

---
## Stage 4 - IDW Interpolation of Missing Occupancy Rates
For parcels with TAU or VHR units but no zone rate, use IDW on neighbouring
known-rate parcels. Any still-null rates after IDW get the basin-wide fallback.

**Inputs:** `parcel_occupancy_rates.parquet`
**Checkpoint:** `parcel_occupancy_interpolated.parquet`

In [ ]:
# Requires ArcGIS Pro + Spatial Analyst extension
from arcpy.sa import Idw
arcpy.CheckOutExtension("Spatial")
sdfParcel = pd.read_parquet(CKPT_OCC_RATES)
# Re-attach geometry for spatial operations
sdfParcel_geo = get_fs_data_spatial_query(PARCEL_URL, query_params=f"Year = {PARCEL_YEAR}")
sdfParcel = sdfParcel.merge(sdfParcel_geo[["APN", "SHAPE"]], on="APN", how="left")
print(f"Loaded {len(sdfParcel):,} parcels with geometry")

In [ ]:
# IDW interpolation function
def idw_fill_nulls(sdf, rate_col, unit_col=None, unit_threshold=0, filter_mask=None):
    '''IDW-interpolate rate_col for parcels where it is null.

    Parcels to interpolate are selected by either:
      - filter_mask  : a boolean Series (takes priority when provided), or
      - unit_col > unit_threshold  : legacy numeric-unit filter.
    '''
    if filter_mask is not None:
        base_mask = filter_mask
    else:
        base_mask = sdf[unit_col].fillna(0) > unit_threshold

    needs_fill = sdf[rate_col].isna() & base_mask
    has_rate   = sdf[rate_col].notna() & base_mask

    if needs_fill.sum() == 0:
        print(f"  {rate_col}: no nulls to fill")
        return sdf
    print(f"  {rate_col}: filling {needs_fill.sum()} null(s) via IDW")
    source_fc  = r"memory\idw_source"
    source_pts = r"memory\idw_source_pts"
    target_fc  = r"memory\idw_target"
    target_pts = r"memory\idw_target_pts"
    out_tbl    = r"memory\idw_values"
    sdf[has_rate][["APN", rate_col, "SHAPE"]].spatial.to_featureclass(source_fc)
    sdf[needs_fill][["APN", "SHAPE"]].spatial.to_featureclass(target_fc)
    # IDW requires point input -- convert parcel polygons to centroids
    arcpy.management.FeatureToPoint(source_fc, source_pts, "CENTROID")
    arcpy.management.FeatureToPoint(target_fc, target_pts, "CENTROID")
    idw_raster = Idw(source_pts, rate_col, cell_size=100, power=2)
    arcpy.sa.ExtractValuesToPoints(target_pts, idw_raster, out_tbl)
    result = pd.DataFrame.spatial.from_featureclass(out_tbl)
    result.columns = [c.upper() for c in result.columns]  # arcpy may change case on round-trip
    result = result[["APN", "RASTERVALU"]].rename(columns={"RASTERVALU": "_interp"})
    sdf = sdf.merge(result, on="APN", how="left")
    fill_mask = sdf[rate_col].isna() & sdf["_interp"].notna()
    sdf.loc[fill_mask, rate_col] = sdf.loc[fill_mask, "_interp"]
    sdf = sdf.drop(columns=["_interp"])
    remaining = sdf[rate_col].isna() & base_mask
    print(f"  {rate_col}: {remaining.sum()} still null after IDW")
    for fc in [source_fc, source_pts, target_fc, target_pts, out_tbl]:
        if arcpy.Exists(fc):
            arcpy.management.Delete(fc)
    return sdf


# TAU: fill nulls for parcels that have tourist accommodation units
sdfParcel = idw_fill_nulls(sdfParcel, "TAU_Occupancy_Rate",
                           unit_col="TouristAccommodation_Units")

# VHR: fill nulls for VHR-flagged parcels (mirrors notebook.ipynb)
sdfParcel = idw_fill_nulls(sdfParcel, "VHR_Occupancy_Rate",
                           filter_mask=(sdfParcel["VHR"] == "Yes"))


In [ ]:
# Apply basin-wide fallbacks for any remaining nulls
tau_null = sdfParcel["TAU_Occupancy_Rate"].isna() & (sdfParcel["TouristAccommodation_Units"].fillna(0) > 0)
vhr_null = sdfParcel["VHR_Occupancy_Rate"].isna() & (sdfParcel["VHR"] == "Yes")
sdfParcel.loc[tau_null, "TAU_Occupancy_Rate"] = DEFAULT_TAU_OCCUPANCY
sdfParcel.loc[vhr_null, "VHR_Occupancy_Rate"] = DEFAULT_VHR_OCCUPANCY
print(f"Fallback TAU ({DEFAULT_TAU_OCCUPANCY}) applied to {tau_null.sum()} parcels")
print(f"Fallback VHR ({DEFAULT_VHR_OCCUPANCY}) applied to {vhr_null.sum()} parcels")

sdfParcel.drop(columns=["SHAPE"], errors="ignore").to_parquet(CKPT_OCC_INTERP, index=False)
print(f"Checkpoint saved: {CKPT_OCC_INTERP}")

### QA - Stage 4

In [ ]:
tau_p = sdfParcel[sdfParcel["TouristAccommodation_Units"].fillna(0) > 0]
vhr_p = sdfParcel[sdfParcel["VHR"] == "Yes"]
print(f"TAU parcels still missing rate: {tau_p['TAU_Occupancy_Rate'].isna().sum()}")
print(f"VHR parcels still missing rate: {vhr_p['VHR_Occupancy_Rate'].isna().sum()}")
print("\nTAU rate distribution:")
print(tau_p["TAU_Occupancy_Rate"].describe().round(3).to_string())

---
## Stage 5 - Census / Socioeconomic Attribution
Map block-group demographics (occupancy, seasonal rate, household size, income)
onto individual parcels, then derive unit-count fields.

**Inputs:** `parcel_occupancy_interpolated.parquet`, `census_block_group_rates_2022.csv` *(frozen 2022 snapshot — see Gotcha #11)*
**Checkpoint:** `parcel_socioeconomic.parquet`

In [ ]:
sdfParcel = pd.read_parquet(CKPT_OCC_INTERP)
print(f"Loaded {len(sdfParcel):,} parcels from checkpoint")

### 5a - Occupancy and seasonal rates from census

### Stage 5 Census Patch — Frozen 2022 Rates (Gotcha \#11)

**Problem:** `Demographics/MapServer/28` schema changed from wide format (one column per
ACS variable) to long format (`variable_code`/`value` rows), and the `year_sample` field was
removed. The server-side `year_sample = 2022` filter is silently ignored, so
`pivot_table(aggfunc="first")` picks up mixed-year records, producing rates that differ
from the 2022 calibration run.

**Fix:** Read block-group rates from `census_block_group_rates_2022.csv` — a frozen
snapshot extracted from `parcel_pickle4.pkl` (the 2022 run's final post-VHR-adjustment
parcel file). The `SecondaryResidence_Rate` column already includes the VHR unit
subtraction, so the VHR adjustment block in Cell 48 is also patched out.

**Calibration note:** The 2025 forecast uses 2022-calibrated model coefficients.
Changing block-group rates would require full re-calibration.

**To revert:** Delete this cell and the patch code cell below, uncomment cells 42, 44, 46,
and uncomment the VHR adjustment block in Cell 48.

In [ ]:
# ── PATCH Stage 5a / 5b / 5c: FROZEN 2022 CENSUS BLOCK-GROUP RATES ─────────
# See patch markdown cell above and Gotcha #11 in base_year_data_engineering_methods.md.

_csv_path   = DATA_DIR / "census_block_group_rates_2022.csv"
_df_frozen  = pd.read_csv(_csv_path)
_df_frozen["TRPAID"] = _df_frozen["BLOCK_GROUP"].astype(str).str.zfill(16)

# Build df_occ_bg (replaces Cell 42)
df_occ_bg = _df_frozen[["TRPAID", "PrimaryResidence_Rate", "SecondaryResidence_Rate"]].copy()
print(f"Frozen occ BG rows     : {len(df_occ_bg)}")

# Build df_hh (replaces Cell 44)
df_hh = _df_frozen[["TRPAID", "PersonsPerUnit"]].copy()
print(f"Frozen hh  BG rows     : {len(df_hh)}")

# Build df_inc_pivot (replaces Cell 46)
df_inc_pivot = _df_frozen[["TRPAID", "HighIncome_Rate", "MediumIncome_Rate", "LowIncome_Rate"]].copy()
print(f"Frozen inc BG rows     : {len(df_inc_pivot)}")
print("PATCH applied: census rates loaded from frozen 2022 CSV (Gotcha #11)")


In [ ]:
# PATCHED OUT -- see patch cells below (Gotcha #11).
# Original code: build df_occ_bg from live ACS service.
# Revert: remove the two comment lines above, uncomment the block below,
#         and delete the patch cells inserted after the Stage 5a header.
#
# df_cen = df_census.copy()
# # Filter to block-group level only 
# if "sample_level" in df_cen.columns:
#     df_cen = df_cen[df_cen["sample_level"] == "block group"].copy()
#     print(f"Census filtered to block group: {len(df_cen):,} records")
# # Normalize TRPAID: strip float ".0" artifact, then zero-pad to 16
# df_cen["TRPAID"] = (df_cen["TRPAID"].astype(str)
#                     .str.strip()
#                     .str.replace(r"\.0$", "", regex=True)
#                     .str.zfill(16))

# # B25002_002E = occupied units, B25002_003E = vacant, B25004_006E = vacant for seasonal use
# occ_vars  = ["B25002_002E", "B25002_003E", "B25004_006E"]
# df_occ_bg = (
#     df_cen[df_cen["variable_code"].isin(occ_vars)]
#     .pivot_table(index="TRPAID", columns="variable_code", values="value", aggfunc="first")
#     .reset_index()
# )
# total_units = df_occ_bg["B25002_002E"].fillna(0) + df_occ_bg["B25002_003E"].fillna(0)
# df_occ_bg["PrimaryResidence_Rate"]   = df_occ_bg["B25002_002E"].fillna(0) / total_units.replace(0, np.nan)
# df_occ_bg["SecondaryResidence_Rate"] = (
#     df_occ_bg["B25004_006E"].fillna(0) / df_occ_bg["B25002_003E"].fillna(0).replace(0, np.nan)
# )
# print(f"Occupancy by block group: {len(df_occ_bg)} rows")
# print(f"Sample census TRPAID: {df_occ_bg['TRPAID'].head(3).tolist()}")


### 5b - Household size

In [ ]:
# PATCHED OUT -- see patch cells below (Gotcha #11).
# Original code: build df_hh from live ACS service.
#
# # B25010_001E = average household size of occupied housing units
# df_hh = (
#     df_cen[df_cen["variable_code"] == "B25010_001E"][["TRPAID", "value"]]
#     .rename(columns={"value": "PersonsPerUnit_Raw"})
#     .copy()
# )
# df_hh["TRPAID"]         = df_hh["TRPAID"].astype(str).str.zfill(16)
# df_hh["PersonsPerUnit"] = df_hh["PersonsPerUnit_Raw"] * HOUSEHOLD_SIZE_ADJUSTMENT
# print(f"Household size records : {len(df_hh)}")
# print(f"Adjustment factor      : {HOUSEHOLD_SIZE_ADJUSTMENT:.6f}")


### 5c - Income distribution

In [ ]:
# PATCHED OUT -- see patch cells below (Gotcha #11).
# Original code: build df_inc_pivot from live ACS service.
#
# df_income_codes = pd.read_csv(LOOKUP_DIR / "income_census_codes.csv")
# df_inc = df_cen[df_cen["variable_code"].isin(df_income_codes["variable_code"])].copy()
# df_inc["TRPAID"] = df_inc["TRPAID"].astype(str).str.zfill(16)
# df_inc = df_inc.merge(df_income_codes[["variable_code", "category"]], on="variable_code", how="left")

# df_inc_pivot = (
#     df_inc.groupby(["TRPAID", "category"])["value"]
#     .sum().unstack(fill_value=0).reset_index()
# )
# inc_total = df_inc_pivot[["High Income", "Medium Income", "Low Income"]].sum(axis=1)
# df_inc_pivot["HighIncome_Rate"]   = df_inc_pivot["High Income"]   / inc_total.replace(0, np.nan)
# df_inc_pivot["MediumIncome_Rate"] = df_inc_pivot["Medium Income"] / inc_total.replace(0, np.nan)
# df_inc_pivot["LowIncome_Rate"]    = df_inc_pivot["Low Income"]    / inc_total.replace(0, np.nan)
# print(f"Income distribution by block group: {len(df_inc_pivot)} rows")


### 5d - Map to parcels and derive unit counts

In [ ]:
# Combine all block-group attribute tables
df_bg = (
    df_occ_bg[["TRPAID", "PrimaryResidence_Rate", "SecondaryResidence_Rate"]]
    .merge(df_hh[["TRPAID", "PersonsPerUnit"]], on="TRPAID", how="outer")
    .merge(df_inc_pivot[["TRPAID", "HighIncome_Rate", "MediumIncome_Rate", "LowIncome_Rate"]], on="TRPAID", how="outer")
)

# BLOCK_GROUP values are now 16-char census TRPAIDs from the 2020 BG filter in
# Stage 2 -- direct match to census TRPAID
overlap = set(sdfParcel["BLOCK_GROUP"].dropna()) & set(df_bg["TRPAID"].dropna())
print(f"Null BLOCK_GROUP        : {sdfParcel['BLOCK_GROUP'].isna().sum()}")
print(f"Join key overlap        : {len(overlap)} of {sdfParcel['BLOCK_GROUP'].nunique()} parcel BGs")

sdfParcel = sdfParcel.merge(df_bg, left_on="BLOCK_GROUP", right_on="TRPAID",
                            how="left", suffixes=("", "_census"))

# Use census values, falling back to any pre-existing parcel values
for col in ["PrimaryResidence_Rate", "SecondaryResidence_Rate", "PersonsPerUnit",
            "HighIncome_Rate", "MediumIncome_Rate", "LowIncome_Rate"]:
    census_col = col + "_census"
    if census_col in sdfParcel.columns:
        sdfParcel[col] = sdfParcel[census_col].combine_first(sdfParcel[col])
        sdfParcel = sdfParcel.drop(columns=[census_col])

# ── PATCH 5d: VHR ADJUSTMENT SKIPPED ──────────────────────────────────────────
# SecondaryResidence_Rate in census_block_group_rates_2022.csv is already the
# VHR-adjusted rate from the 2022 run (extracted from parcel_pickle4.pkl).
# Re-running the adjustment would double-subtract VHR units.
# Correct code preserved below (commented); revert by removing this note block
# and uncommenting the lines below.
#
# # --- VHR-adjusted secondary residence rate (mirrors notebook.ipynb) ---
# # VHR units are already counted as seasonal in the census; subtract them from
# # the seasonal unit estimate per block group to avoid double-counting.
# vhrs = sdfParcel.loc[sdfParcel["VHR"] == "Yes"]

# totalRes = (
#     sdfParcel
#     .groupby("BLOCK_GROUP", as_index=False)
#     .agg(Residential_Units_Adjusted=("Residential_Units_Adjusted", "sum"),
#          PrimaryResidence_Rate      =("PrimaryResidence_Rate",       "mean"),
#          SecondaryResidence_Rate    =("SecondaryResidence_Rate",     "mean"))
# )
# totalVHR = (
#     vhrs
#     .groupby("BLOCK_GROUP", as_index=False)
#     .agg(VHR_Units=("Residential_Units_Adjusted", "sum"))
# )

# totalResVHR = totalRes.merge(totalVHR, on="BLOCK_GROUP", how="left")
# totalResVHR["VHR_Units"] = totalResVHR["VHR_Units"].fillna(0)

# res_bg = totalResVHR["Residential_Units_Adjusted"]
# totalResVHR["non_primary_residence_units"] = res_bg - (totalResVHR["PrimaryResidence_Rate"] * res_bg)
# # SecondaryResidence_Rate = seasonal_vacant / total_vacant; apply to vacant units (non-primary)
# # to correctly estimate seasonal units without inflating by total/vacant ratio.
# totalResVHR["non_adjusted_seasonal_units"] = totalResVHR["SecondaryResidence_Rate"] * totalResVHR["non_primary_residence_units"]
# totalResVHR["adjusted_seasonal_units"]     = totalResVHR["non_adjusted_seasonal_units"] - totalResVHR["VHR_Units"]

# # Beach Club data lag -- manually zero this block group
# totalResVHR.loc[totalResVHR["BLOCK_GROUP"] == "3200500170022020", "adjusted_seasonal_units"] = 0

# totalResVHR["adjusted_seasonal_rate"] = (
#     totalResVHR["adjusted_seasonal_units"]
#     / totalResVHR["non_primary_residence_units"].replace(0, np.nan)
# )
# sdfParcel["SecondaryResidence_Rate"] = sdfParcel["BLOCK_GROUP"].map(
#     totalResVHR.set_index("BLOCK_GROUP")["adjusted_seasonal_rate"]
# )

# Derive unit count fields
res = sdfParcel["Residential_Units_Adjusted"].fillna(0)
sdfParcel["OccupiedUnits"]   = res * sdfParcel["PrimaryResidence_Rate"].fillna(0)
sdfParcel["UnoccupiedUnits"] = res - sdfParcel["OccupiedUnits"]
sdfParcel["SeasonalUnits"]   = sdfParcel["UnoccupiedUnits"] * sdfParcel["SecondaryResidence_Rate"].fillna(0)
sdfParcel["HighUnits"]       = sdfParcel["OccupiedUnits"] * sdfParcel["HighIncome_Rate"].fillna(0)
sdfParcel["MediumUnits"]     = sdfParcel["OccupiedUnits"] * sdfParcel["MediumIncome_Rate"].fillna(0)
sdfParcel["LowUnits"]        = sdfParcel["OccupiedUnits"] * sdfParcel["LowIncome_Rate"].fillna(0)
sdfParcel["People"]          = sdfParcel["OccupiedUnits"] * sdfParcel["PersonsPerUnit"].fillna(0)

sdfParcel.drop(columns=["TRPAID"], errors="ignore", inplace=True)


### 5e - Rounding and integer validation

In [ ]:
# Fillna unit columns so nulls don't propagate into TAZ sums
for col in ["OccupiedUnits", "SeasonalUnits", "HighUnits", "MediumUnits", "LowUnits", "People"]:
    sdfParcel[col] = sdfParcel[col].fillna(0)

### QA: check for any parcels with occupied units but zero population, which would indicate a problem with the PersonsPerUnit rate or the occupancy rates that feed into it. This is a common issue in synthetic population generation and can lead to underestimation of people counts if not addressed. The fix is to set People to zero for any parcel that has no occupied units, regardless of the PersonsPerUnit value.
# # Zero out People where there are no occupied units
# pop_fix = (sdfParcel["People"] > 0) & (sdfParcel["OccupiedUnits"] == 0)
# if pop_fix.sum() > 0:
#     print(f"  Population zeroed for {pop_fix.sum()} parcels with 0 occupied units")
#     sdfParcel.loc[pop_fix, "People"] = 0
# else:
#     print("  Population fix: no parcels affected")

sdfParcel.to_parquet(CKPT_SOCIOECONOMIC, index=False)
print(f"Checkpoint saved: {CKPT_SOCIOECONOMIC}")

# Export as zipped File Geodatabase for sharing
import geopandas as gpd
import zipfile
import shutil
from shapely import wkb as shapely_wkb

sdfParcel_geo = get_fs_data_spatial_query(PARCEL_URL, query_params=f"Year = {PARCEL_YEAR}")
sdfParcel_with_geom = sdfParcel.merge(sdfParcel_geo[["APN", "SHAPE"]], on="APN", how="left")

gdf = gpd.GeoDataFrame(
    sdfParcel_with_geom[FINAL_SCHEMA],
    geometry=sdfParcel_with_geom["SHAPE"].apply(
        lambda g: shapely_wkb.loads(bytes(g.WKB)) if g else None
    ),
    crs="EPSG:26910"
)

gdb_path = OUT_DIR / "parcels_base_year.gdb"
gdb_zip_path = OUT_DIR / "parcels_base_year.gdb.zip"

# Remove stale GDB folder if it exists
if gdb_path.exists():
    shutil.rmtree(gdb_path)

gdf.to_file(gdb_path, driver="OpenFileGDB", layer="parcels_base_year")

# Zip the .gdb folder so it can be shared as a single file
with zipfile.ZipFile(gdb_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in gdb_path.rglob("*"):
        zf.write(fp, fp.relative_to(OUT_DIR))

print(f"File Geodatabase saved: {gdb_zip_path}")


### QA - Stage 5

In [ ]:
qa_summary(sdfParcel, "After Socioeconomic Attribution",
    unit_cols=["Residential_Units_Adjusted", "OccupiedUnits", "SeasonalUnits", "People"])
print(f"  PrimaryResidence_Rate nulls  : {sdfParcel['PrimaryResidence_Rate'].isna().sum()}")
print(f"  SecondaryResidence_Rate nulls: {sdfParcel['SecondaryResidence_Rate'].isna().sum()}")
print(f"  PersonsPerUnit nulls         : {sdfParcel['PersonsPerUnit'].isna().sum()}")

---
## Stage 6 - Campground Processing
Aggregate campground overnight capacity to TAZ level.

**Inputs:** `sdf_campgrounds`, `df_campvisits`
**Checkpoint:** `campground_taz.parquet`

In [ ]:
# [ARCPY] Spatial join campground locations to TAZ
camp_taz_lookup = arcpy_spatial_join_attr(
    sdf_campgrounds, sdf_taz, "TAZ", key_field="RECREATION_NAME"
)
sdf_campgrounds["TAZ"] = sdf_campgrounds["RECREATION_NAME"].map(camp_taz_lookup)
print(f"Campground TAZ nulls: {sdf_campgrounds['TAZ'].isna().sum()}")

In [ ]:
df_camp = sdf_campgrounds.merge(df_campvisits, left_on="RECREATION_NAME", right_on="Campground", how="left")
df_camp = df_camp[df_camp["Campground"] != "Bayview Campground"].copy()

# Compute sites sold: Total_Sites * Occupancy_Rate (used for occupancy rate output only)
df_camp["SitesSold"] = df_camp["Total_Sites"] * df_camp["Occupancy_Rate"]

df_camp_taz = (
    df_camp.groupby("TAZ", as_index=False)
    .agg(Total_Sites=("Total_Sites", "sum"),
         campground  =("Total_Sites", "sum"))   # campground = Total_Sites (capacity), matching 2022 logic
)

df_camp_taz.to_parquet(CKPT_CAMPGROUND, index=False)
print(f"TAZs with campgrounds : {len(df_camp_taz)}")
print(f"Total campground sites: {df_camp_taz['Total_Sites'].sum():,}")
print(f"Total campground sites sold: {df_camp['SitesSold'].sum():,.0f}")


---
## Stage 7 - School Enrollment
Spatial-join schools to TAZ, then aggregate enrollment counts.

**Inputs:** `df_school_table`, `sdf_schools`, `sdf_taz`
**Checkpoint:** `school_enrollment.parquet`  |  **Output:** `SchoolEnrollment.csv`

In [ ]:
# [ARCPY] Spatial join school locations to TAZ (key on NAME to match reference)
school_taz_lookup = arcpy_spatial_join_attr(sdf_schools, sdf_taz, "TAZ", key_field="NAME")

# Classify each school by type based on its name 
sdf_schools_typed = sdf_schools.copy()
sdf_schools_typed["TYPE"] = None
sdf_schools_typed.loc[sdf_schools_typed["NAME"].str.contains("elementary", case=False), "TYPE"] = "Elementary School"
sdf_schools_typed.loc[sdf_schools_typed["NAME"].str.contains("middle",     case=False), "TYPE"] = "Middle School"
sdf_schools_typed.loc[sdf_schools_typed["NAME"].str.contains("high",       case=False), "TYPE"] = "High School"
sdf_schools_typed.loc[sdf_schools_typed["NAME"].str.contains("college",    case=False), "TYPE"] = "College"
# Default unclassified schools to Elementary School
sdf_schools_typed.loc[sdf_schools_typed["TYPE"].isnull(), "TYPE"] = "Elementary School"

# Map TAZ onto each school point; normalise to numeric to match sdf_taz dtype
sdf_schools_typed["TAZ"] = pd.to_numeric(
    sdf_schools_typed["NAME"].map(school_taz_lookup), errors="coerce"
)

school_grouped = (
    sdf_schools_typed.dropna(subset=["TAZ"])
    .groupby(["TYPE", "TAZ"], as_index=False)["ENROLLMENT"]
    .sum()
)

# Pivot: one row per TAZ, one column per school type
school_pivot = school_grouped.pivot(index="TAZ", columns="TYPE", values="ENROLLMENT").reset_index()
school_pivot.columns.name = None
school_pivot["TAZ"] = pd.to_numeric(school_pivot["TAZ"], errors="coerce")

# Merge with full TAZ list (cast both keys to numeric to avoid dtype conflict)
taz_frame = sdf_taz[["TAZ"]].copy()
taz_frame["TAZ"] = pd.to_numeric(taz_frame["TAZ"], errors="coerce")
df_school_taz = pd.merge(taz_frame, school_pivot, on="TAZ", how="left")

# Rename to standard column names
df_school_taz.rename(columns={
    "Elementary School": "elementary_school_enrollment",
    "Middle School":     "middle_school_enrollment",
    "High School":       "high_school_enrollment",
    "College":           "college_enrollment",
}, inplace=True)

# Ensure all four columns exist even if no schools of that type were found
for col in ["elementary_school_enrollment", "middle_school_enrollment",
            "high_school_enrollment", "college_enrollment"]:
    if col not in df_school_taz.columns:
        df_school_taz[col] = 0

enroll_cols = ["elementary_school_enrollment", "middle_school_enrollment",
               "high_school_enrollment", "college_enrollment"]

df_school_taz = (
    df_school_taz.groupby("TAZ", as_index=False)[enroll_cols]
    .sum()
    .fillna(0)
    .astype({c: int for c in enroll_cols})
    .astype({"TAZ": int})
)

# Add phantom TAZs that have no polygons but are required by the model (all-zero enrollment)
PHANTOM_TAZS = [1, 2, 3, 4, 5, 6, 7, 10, 20, 30, 40, 50, 60, 70]
existing_school_tazs = set(df_school_taz["TAZ"].tolist())
phantom_rows = pd.DataFrame({
    "TAZ": [t for t in PHANTOM_TAZS if t not in existing_school_tazs],
    **{col: 0 for col in enroll_cols}
})
if len(phantom_rows):
    df_school_taz = (
        pd.concat([df_school_taz, phantom_rows], ignore_index=True)
        .sort_values("TAZ").reset_index(drop=True)
    )
    print(f"  Added {len(phantom_rows)} phantom TAZs with zero enrollment.")

# ── PATCH: Phantom TAZ college enrollment hardcodes (Gotcha #12) ────────────
# TAZs 1 and 2 have college enrollment that is not captured by the spatial
# school join because the campus records do not fall inside their TAZ polygons.
# Values sourced from reference enrollment data and confirmed against 2022 run.
_college_overrides = {1: 171, 2: 49}
for _taz, _val in _college_overrides.items():
    _mask = df_school_taz["TAZ"] == _taz
    if _mask.any():
        df_school_taz.loc[_mask, "college_enrollment"] = _val
        print(f"  PATCH: TAZ {_taz} college_enrollment = {_val}")
# ─────────────────────────────────────────────────────────────────────────────

df_school_taz.to_parquet(CKPT_SCHOOL, index=False)
df_school_taz.to_csv(OUT_DIR / f"SchoolEnrollment.csv", index=False)
print(f"School checkpoint saved.")
print(f"Total enrollment: {df_school_taz[enroll_cols].sum().sum():,}")

---
## Stage 8 - TAZ Aggregation & Output CSV Export
Load all checkpoints and build the five travel-demand model input files.

| Output file | Contents |
|---|---|
| `OvernightVisitorZonalData_Summer.csv` | TAU rooms by type, campground sites, % seasonal |
| `VisitorOccupancyRates_Summer.csv`     | Weighted TAU and VHR occupancy rates by TAZ |
| `SocioEcon_Summer.csv`                 | Occupancy, seasonal, household size, income shares |
| `Employment.csv`                       | Employment by sector and TAZ |
| `inputs_summarized.csv`                | All of the above joined into one wide table |

### 8a - Load checkpoints

In [ ]:
sdfParcel   = pd.read_parquet(CKPT_SOCIOECONOMIC)
df_camp_taz = pd.read_parquet(CKPT_CAMPGROUND)
df_school   = pd.read_parquet(CKPT_SCHOOL)
print(f"Parcels      : {len(sdfParcel):,}")
print(f"Camp TAZ rows: {len(df_camp_taz)}")
print(f"School rows  : {len(df_school)}")

### 8b - Overnight visitor zonal data

In [ ]:
# TAU rooms by type aggregated to TAZ
df_tau_taz = (
    sdfParcel[sdfParcel["TouristAccommodation_Units"].fillna(0) > 0]
    .groupby(["TAZ", "TAU_TYPE"])["TouristAccommodation_Units"]
    .sum().unstack(fill_value=0).reset_index()
)
for col in ["HotelMotel", "Casino", "Resort"]:
    if col not in df_tau_taz.columns:
        df_tau_taz[col] = 0
df_tau_taz = df_tau_taz.rename(columns={"HotelMotel": "hotelmotel", "Casino": "casino", "Resort": "resort"})

# percentHouseSeasonal = mean(SecondaryResidence_Rate) per TAZ
# This is the VHR-adjusted seasonal rate, matching 2022 notebook cell 57:
# df_overnight.percentHouseSeasonal mapped from sdfParcel_taz[SecondaryResidence_Rate]
df_seasonal = (
    sdfParcel.groupby("TAZ", as_index=False)
    .agg(percentHouseSeasonal=("SecondaryResidence_Rate", "mean"))
)


df_overnight = (
    df_tau_taz
    .merge(df_camp_taz, on="TAZ", how="outer")
    .merge(df_seasonal[["TAZ", "percentHouseSeasonal"]], on="TAZ", how="outer")
    .fillna(0)
)
for col in ["hotelmotel", "casino", "resort", "campground"]:
    if col in df_overnight.columns:
        df_overnight[col] = df_overnight[col].astype(int)

# rec_attractiveness: copied from example input file (not recomputed for this run)
df_rec = example_dfs["OvernightVisitorZonalData_Summer.csv"][["taz", "rec_attractiveness"]].copy()
df_rec["taz"] = pd.to_numeric(df_rec["taz"], errors="coerce")

# Match target schema: taz (lowercase), column order matches Example_Input_File_Structure
df_overnight = df_overnight.rename(columns={"TAZ": "taz"})
df_overnight["taz"] = pd.to_numeric(df_overnight["taz"], errors="coerce")
df_overnight = df_overnight.merge(df_rec, on="taz", how="left")
df_overnight["rec_attractiveness"] = df_overnight["rec_attractiveness"].fillna(0).astype(int)
df_overnight = df_overnight[["taz", "hotelmotel", "resort", "casino", "campground",
                               "percentHouseSeasonal", "rec_attractiveness"]]

# TAZs 263, 356, 357 are rural with no development â€” parcel centroids may not fall inside
# them, causing them to drop from the aggregation. Add them back as all-zero rows.
REQUIRED_OVERNIGHT_TAZS = [263, 355, 356, 357]
existing_overnight_tazs = set(df_overnight["taz"].tolist())
missing_overnight = [
    {"taz": t, "hotelmotel": 0, "resort": 0, "casino": 0, "campground": 0,
     "percentHouseSeasonal": 0.0, "rec_attractiveness": 0}
    for t in REQUIRED_OVERNIGHT_TAZS if t not in existing_overnight_tazs
]
if missing_overnight:
    df_overnight = (
        pd.concat([df_overnight, pd.DataFrame(missing_overnight)], ignore_index=True)
        .sort_values("taz").reset_index(drop=True)
    )
    print(f"  Added {len(missing_overnight)} missing TAZs to OvernightVisitorZonalData.")

df_overnight.to_csv(OUT_DIR / f"OvernightVisitorZonalData_Summer.csv", index=False)
print(f"OvernightVisitorZonalData_Summer.csv -- {len(df_overnight)} TAZs")
df_overnight.head()

### 8c - Visitor occupancy rates

In [ ]:
# Per-type occupancy rates aggregated to TAZ
# Target schema: taz, hotelmotel, resort, casino, campground, house, seasonal

tau_sdf = sdfParcel[sdfParcel["TouristAccommodation_Units"].fillna(0) > 0].copy()
tau_sdf["weighted_rate"] = tau_sdf["TAU_Occupancy_Rate"] * tau_sdf["TouristAccommodation_Units"]

# Weighted average rate per TAU type per TAZ
df_tau_type_occ = (
    tau_sdf.groupby(["TAZ", "TAU_TYPE"])
    .agg(rate_sum=("weighted_rate", "sum"), units=("TouristAccommodation_Units", "sum"))
    .assign(rate=lambda d: d["rate_sum"] / d["units"].replace(0, np.nan))
    .reset_index()[["TAZ", "TAU_TYPE", "rate"]]
    .pivot(index="TAZ", columns="TAU_TYPE", values="rate")
    .reset_index()
)
df_tau_type_occ.columns.name = None
for col in ["HotelMotel", "Resort", "Casino"]:
    if col not in df_tau_type_occ.columns:
        df_tau_type_occ[col] = 0.0
df_tau_type_occ = df_tau_type_occ.rename(columns={
    "HotelMotel": "hotelmotel",
    "Resort":     "resort",
    "Casino":     "casino",
})
df_tau_type_occ["TAZ"] = pd.to_numeric(df_tau_type_occ["TAZ"], errors="coerce")

# ── CORRECTED VHR RATE LOOKUP ────────────────────────────────────────────────────────────
# Technically correct: per-TAZ mean of VHR_Occupancy_Rate from the zone
# lookup. The CSLT_ALL override applied in Stage 2e correctly aggregates all
# CSLT VHR parcels to a single zone, fixing a bug in the 2022 run where those
# parcels fell through to the basin-wide default (0.4223) because CSLT_Zone1-5
# have no VHR row in occupancy_rates.csv. The correct CSLT-wide VHR rate is
# 0.5258. See Gotcha #10 in base_year_data_engineering_methods.md.
#
# PATCHED OUT -- see patch cells below. Remove the comments here and delete
# the patch cells to restore the corrected rates after re-calibration.
# ───────────────────────────────────────────────────────────────────────────────
# df_vhr_occ = (
#     sdfParcel[sdfParcel["VHR"] == "Yes"]
#     .groupby("TAZ", as_index=False)
#     .agg(house=("VHR_Occupancy_Rate", "mean"))
# )
# df_vhr_occ["TAZ"] = pd.to_numeric(df_vhr_occ["TAZ"], errors="coerce")
# # Seasonal: VHR_Occupancy_Rate is the correct proxy for how often seasonal
# # second homes are occupied in summer. SecondaryResidence_Rate is a census
# # vacancy-fraction, not a measured occupancy rate.
# df_seasonal_occ = (
#     sdfParcel[sdfParcel["VHR"] == "Yes"]
#     .groupby("TAZ", as_index=False)
#     .agg(seasonal=("VHR_Occupancy_Rate", "mean"))
# )
# df_seasonal_occ["TAZ"] = pd.to_numeric(df_seasonal_occ["TAZ"], errors="coerce")


### 8c-patch — house/seasonal rates: 2022 calibration compatibility

The travel demand model was **calibrated on 2022 inputs**. The calibration
links observed trip counts to the socioeconomic inputs at that point in time,
including the per-TAZ `house` and `seasonal` VHR occupancy rates. These inputs
are a **2025 forecast** built on top of a model calibrated to 2022: changing
the rates the model was tuned against would shift implied seasonal visitor
demand without a corresponding re-calibration.

The corrected rates from the zone lookup above (~0.526 for CSLT TAZs) differ
from the values those TAZs received in the 2022 run (~0.422, the basin-wide
default they fell through to because CSLT_Zone1–5 have no VHR row in
`occupancy_rates.csv`). That was a bug in 2022, but the calibrated model
coefficients are anchored to those values.

Until re-calibration is performed against the corrected rates, `house` and
`seasonal` are overridden with the per-TAZ values from the 2022 run. The
correct lookup is preserved above (commented out) as a record of the
underlying data quality improvement.

**To revert:** delete this markdown cell and the patch code cell below,
then uncomment the corrected VHR lookup in the cell above.

In [ ]:
# ── PATCH: house/seasonal rates matched to 2022 calibration baseline ────────
# Loads per-TAZ house and seasonal rates from the 2022 base year output and
# substitutes them in place of the corrected CSLT_ALL zone rates computed
# above. This keeps forecast inputs consistent with calibrated model
# coefficients. See markdown cell above and Gotcha #10 in the methods doc.
# ─────────────────────────────────────────────────────────────────────────────

DIR_22_OCC = OUT_DIR.parents[3] / "2022" / "data" / "processed_data"
df_occ_22  = pd.read_csv(DIR_22_OCC / "VisitorOccupancyRates_Summer.csv")
df_occ_22["taz"] = pd.to_numeric(df_occ_22["taz"], errors="coerce")

df_vhr_occ = (
    df_occ_22.loc[df_occ_22["house"] > 0, ["taz", "house"]]
    .rename(columns={"taz": "TAZ"})
    .reset_index(drop=True)
)
df_seasonal_occ = (
    df_occ_22.loc[df_occ_22["seasonal"] > 0, ["taz", "seasonal"]]
    .rename(columns={"taz": "TAZ"})
    .reset_index(drop=True)
)

corrected_mean = (
    sdfParcel[sdfParcel["VHR"] == "Yes"]
    .groupby("TAZ")["VHR_Occupancy_Rate"].mean().mean()
)
print("PATCH applied: house/seasonal rates loaded from 2022 reference")
print(f"  TAZs with nonzero house rate     : {len(df_vhr_occ)}")
print(f"  2022 avg house rate (in use)     : {df_vhr_occ['house'].mean():.4f}")
print(f"  Corrected avg rate (CSLT_ALL, not used): {corrected_mean:.4f}")


In [ ]:
# campground occupancy: copied from example input file (not recomputed for this run)
df_camp_rate = example_dfs["VisitorOccupancyRates_Summer.csv"][["taz", "campground"]].copy()
df_camp_rate = df_camp_rate.rename(columns={"taz": "TAZ"})
df_camp_rate["TAZ"] = pd.to_numeric(df_camp_rate["TAZ"], errors="coerce")

df_visitor_occ = (
    df_tau_type_occ
    .merge(df_camp_rate,    on="TAZ", how="outer")
    .merge(df_vhr_occ,      on="TAZ", how="outer")
    .merge(df_seasonal_occ, on="TAZ", how="outer")
    .fillna(0)
)

# Match target schema: taz (lowercase), column order
df_visitor_occ = df_visitor_occ.rename(columns={"TAZ": "taz"})
df_visitor_occ = df_visitor_occ[["taz", "hotelmotel", "resort", "casino",
                                   "campground", "house", "seasonal"]]
df_visitor_occ.to_csv(OUT_DIR / "VisitorOccupancyRates_Summer.csv", index=False)
print(f"VisitorOccupancyRates_Summer.csv -- {len(df_visitor_occ)} TAZs")
df_visitor_occ.head()


### 8d - Socioeconomic data

In [ ]:
# Employment by sector per TAZ (does not change from base year)
employment_source = OUT_DIR / f"Employment.csv"
fallback_employ   = OUT_DIR.parents[0] / "processed_data" / "Archive" / "Employment.csv"

if employment_source.exists():
    df_employ_raw = pd.read_csv(employment_source)
elif fallback_employ.exists():
    df_employ_raw = pd.read_csv(fallback_employ)
else:
    print(f"WARNING: Employment CSV not found â€” employment columns will be 0")
    df_employ_raw = pd.DataFrame()

if len(df_employ_raw):
    taz_col = "taz" if "taz" in df_employ_raw.columns else "TAZ"
    emp_keep = [c for c in [taz_col, "emp_retail", "emp_srvc", "emp_rec", "emp_gaming", "emp_other"]
                if c in df_employ_raw.columns]
    df_employ_taz = (
        df_employ_raw[emp_keep]
        .rename(columns={taz_col: "TAZ", "emp_gaming": "emp_game"})
        .groupby("TAZ", as_index=False).sum()
    )
else:
    df_employ_taz = pd.DataFrame()

# Socioeconomic aggregation to TAZ
df_socio_base = (
    sdfParcel.groupby("TAZ", as_index=False)
    .agg(
        total_residential_units  =("Residential_Units_Adjusted", "sum"),
        census_occ_rate          =("PrimaryResidence_Rate",      "mean"),
        total_occ_units          =("OccupiedUnits",              "sum"),
        low_inc_rate             =("LowIncome_Rate",             "mean"),
        med_inc_rate             =("MediumIncome_Rate",          "mean"),
        high_inc_rate            =("HighIncome_Rate",            "mean"),
        persons_per_occ_unit     =("PersonsPerUnit",             "mean"),
        total_persons            =("People",                     "sum"),
    )
    .fillna(0)
)

# Derive income unit counts from proportions x total occupied units
df_socio_base["occ_units_low_inc"]  = (df_socio_base["low_inc_rate"]  * df_socio_base["total_occ_units"]).round().astype(int)
df_socio_base["occ_units_med_inc"]  = (df_socio_base["med_inc_rate"]  * df_socio_base["total_occ_units"]).round().astype(int)
df_socio_base["occ_units_high_inc"] = (df_socio_base["high_inc_rate"] * df_socio_base["total_occ_units"]).round().astype(int)
df_socio_base = df_socio_base.drop(columns=["low_inc_rate", "med_inc_rate", "high_inc_rate"])

# Merge employment by TAZ
if len(df_employ_taz):
    df_employ_taz["TAZ"] = pd.to_numeric(df_employ_taz["TAZ"], errors="coerce")
    df_socio_base["TAZ"] = pd.to_numeric(df_socio_base["TAZ"], errors="coerce")
    df_socio = df_socio_base.merge(df_employ_taz, on="TAZ", how="left").fillna(0)
    for ecol in ["emp_retail", "emp_srvc", "emp_rec", "emp_game", "emp_other"]:
        if ecol not in df_socio.columns:
            df_socio[ecol] = 0
else:
    df_socio = df_socio_base.copy()
    for ecol in ["emp_retail", "emp_srvc", "emp_rec", "emp_game", "emp_other"]:
        df_socio[ecol] = 0

# Enforce integer types on all count columns (floats after fillna/merge)
int_cols = [
    "total_residential_units", "total_occ_units",
    "occ_units_low_inc", "occ_units_med_inc", "occ_units_high_inc",
    "total_persons",
    "emp_retail", "emp_srvc", "emp_rec", "emp_game", "emp_other",
]
df_socio[int_cols] = df_socio[int_cols].round().astype(int)

# Match target schema: taz (lowercase), column order matches Example_Input_File_Structure
df_socio = df_socio.rename(columns={"TAZ": "taz"})
df_socio = df_socio[[
    "taz", "total_residential_units", "census_occ_rate", "total_occ_units",
    "occ_units_low_inc", "occ_units_med_inc", "occ_units_high_inc",
    "persons_per_occ_unit", "total_persons",
    "emp_retail", "emp_srvc", "emp_rec", "emp_game", "emp_other"
]]

# TAZs 263, 355, 356, 357 are rural with no development â€” parcel centroids may not fall
# inside them, causing them to drop from the aggregation. Add them back as all-zero rows.
REQUIRED_SOCIO_TAZS = [263, 355, 356, 357]
existing_socio_tazs = set(df_socio["taz"].tolist())
missing_socio = [
    {"taz": t, **{col: 0 for col in df_socio.columns if col != "taz"}}
    for t in REQUIRED_SOCIO_TAZS if t not in existing_socio_tazs
]
if missing_socio:
    df_socio = (
        pd.concat([df_socio, pd.DataFrame(missing_socio)], ignore_index=True)
        .sort_values("taz").reset_index(drop=True)
    )
    print(f"  Added {len(missing_socio)} missing TAZs to SocioEcon_Summer.")

df_socio.to_csv(OUT_DIR / f"SocioEcon_Summer.csv", index=False)
print(f"SocioEcon_Summer.csv -- {len(df_socio)} TAZs, {len(df_socio.columns)} columns")
df_socio.head()

### 8e - Employment
Employment is carried forward unchanged from the 2022 base year.
The source file lives in the same `processed_data` folder from the prior run.
Update `EMPLOYMENT_SOURCE` in Config if the path ever changes.

In [ ]:
# Employment loading and merging is handled in cell 8d (SocioEcon) above.
# The emp_retail, emp_srvc, emp_rec, emp_game, emp_other columns are now
# part of SocioEcon_Summer.csv to match the target schema.
#
# df_employ_taz is still available here for standalone diagnostics.
if 'df_employ_taz' in dir() and len(df_employ_taz):
    print(f"Employment loaded: {len(df_employ_taz)} TAZs")
    print(df_employ_taz.head())
else:
    print("Employment data not available.")


### 8f - Combined inputs summary

In [ ]:
# Normalize TAZ dtype across all frames before merging
# Rename lowercase taz -> TAZ for consistent merge key, then cast to Int64
for _df in [df_socio, df_overnight, df_visitor_occ, df_school, df_employ_taz]:
    if not len(_df):
        continue
    if "taz" in _df.columns and "TAZ" not in _df.columns:
        _df.rename(columns={"taz": "TAZ"}, inplace=True)
    if "TAZ" in _df.columns:
        _df["TAZ"] = pd.to_numeric(_df["TAZ"], errors="coerce").astype("Int64")

school_cols = ["TAZ"] + [c for c in df_school.columns if c != "TAZ"]

df_summary = (
    df_socio
    .merge(df_overnight,           on="TAZ", how="outer")
    .merge(df_visitor_occ,         on="TAZ", how="outer")
    .merge(df_school[school_cols], on="TAZ", how="outer")
)
if len(df_employ_taz):
    df_summary = df_summary.merge(df_employ_taz, on="TAZ", how="outer")

df_summary = df_summary.fillna(0).sort_values("TAZ").reset_index(drop=True)
df_summary.to_csv(OUT_DIR / f"inputs_summarized.csv", index=False)
print(f"inputs_summarized.csv -- {len(df_summary)} TAZs, {len(df_summary.columns)} columns")
df_summary.head()

---
## Stage 9 - QA & Basin Summary
Print basin-wide totals for sanity-checking against prior base years.

In [ ]:
print("=" * 55)
print(f"  BASE YEAR SUMMARY ({PARCEL_YEAR})")
print("=" * 55)
print(f"  Total parcels                : {len(sdfParcel):,}")
print(f"  Total residential units      : {sdfParcel['Residential_Units_Adjusted'].sum():,.0f}")
print(f"  Total TAU rooms              : {sdfParcel['TouristAccommodation_Units'].sum():,.0f}")
print(f"  VHR parcels                  : {(sdfParcel['VHR']=='Yes').sum():,}")
print(f"  Total occupied units         : {sdfParcel['OccupiedUnits'].sum():,.0f}")
print(f"  Total seasonal units         : {sdfParcel['SeasonalUnits'].sum():,.0f}")
print(f"  Total residents (people)     : {sdfParcel['People'].sum():,.0f}")
print(f"  Avg TAU occupancy rate       : {sdfParcel.loc[sdfParcel['TouristAccommodation_Units'].fillna(0)>0,'TAU_Occupancy_Rate'].mean():.4f}")
print(f"  Avg VHR occupancy rate       : {sdfParcel.loc[sdfParcel['VHR']=='Yes','VHR_Occupancy_Rate'].mean():.4f}")
print(f"  Avg household size (adjusted): {sdfParcel['PersonsPerUnit'].mean():.4f}")
print("=" * 55)

In [ ]:
# Check for parcels with no TAZ assignment
missing_taz = sdfParcel["TAZ"].isna().sum()
if missing_taz:
    print(f"WARNING: {missing_taz} parcels have no TAZ assignment")
    print(sdfParcel[sdfParcel["TAZ"].isna()][["APN","JURISDICTION","Residential_Units"]].head(20))
else:
    print("All parcels have TAZ assignments.")

# Confirm all output files exist
output_files = [
    "OvernightVisitorZonalData_Summer.csv",
    "VisitorOccupancyRates_Summer.csv",
    "SocioEcon_Summer.csv",
    "Employment.csv",
    "SchoolEnrollment.csv",
    "inputs_summarized.csv",
]
print("\nOutput files:")
for f in output_files:
    p = OUT_DIR / f
    status = f"{p.stat().st_size:,} bytes" if p.exists() else "MISSING"
    print(f"  {f:<45} {status}")

### Stage 9c - Seasonal visitor input QA
Checks the two inputs that directly drive modeled seasonal and day visitors,
comparing against 2022 base year reference values.

| Input | What it controls | 2022 reference |
|---|---|---|
| `percentHouseSeasonal` | Share of vacant units that are seasonal per TAZ | mean 0.73 |
| `seasonal` occ rate | Fraction of seasonal homes occupied on a given summer day | mean 0.46 |

Pre-fix bugs: `percentHouseSeasonal` was ~0.35 (seasonal/total instead of SecondaryResidence_Rate)
and seasonal occ rate was ~0.87 (a census vacancy fraction, not a measured occupancy rate).


In [ ]:
# Seasonal visitor input QA -----------------------------------------------
# Reference: 2022 base year (3 runs had percentHouseSeasonal of 0.71, 0.79, 0.73)
REF_PHS_MEAN  = 0.73   # mean percentHouseSeasonal across nonzero TAZs
REF_OCC_MEAN  = 0.46   # mean seasonal occupancy rate across nonzero TAZs

df_ov  = pd.read_csv(OUT_DIR / "OvernightVisitorZonalData_Summer.csv")
df_occ = pd.read_csv(OUT_DIR / "VisitorOccupancyRates_Summer.csv")

phs_nonzero = df_ov["percentHouseSeasonal"].replace(0, np.nan).dropna()
s_occ       = df_occ["seasonal"].replace(0, np.nan).dropna()
h_occ       = df_occ["house"].replace(0, np.nan).dropna()

phs_mean = phs_nonzero.mean()
occ_mean = s_occ.mean()

checks = [
    (
        "percentHouseSeasonal mean in range [0.50, 0.90]",
        0.50 <= phs_mean <= 0.90,
        f"{phs_mean:.4f}  (2022 ref: {REF_PHS_MEAN}, pre-fix was ~0.35)"
    ),
    (
        "seasonal occ rate mean in range [0.37, 0.57]",
        0.37 <= occ_mean <= 0.57,
        f"{occ_mean:.4f}  (2022 ref: {REF_OCC_MEAN}, pre-fix was ~0.87)"
    ),
    (
        "seasonal occ rate is NOT a vacancy fraction (must be < 0.75)",
        occ_mean < 0.75,
        f"{occ_mean:.4f}  -- if >=0.75, SecondaryResidence_Rate crept back in"
    ),
    (
        "house occ rate ~= seasonal occ rate (both sourced from VHR_Occupancy_Rate)",
        abs(h_occ.mean() - occ_mean) < 0.02,
        f"house={h_occ.mean():.4f}, seasonal={occ_mean:.4f}"
    ),
]

print(f"{'CHECK':<60} {'PASS/FAIL':<10} VALUE")
print("-" * 105)
all_pass = True
for desc, result, val in checks:
    status = "PASS" if result else "FAIL"
    if not result:
        all_pass = False
    print(f"{desc:<60} {status:<10} {val}")

print()
if all_pass:
    print("All seasonal QA checks passed.")
else:
    print("WARNING: One or more seasonal QA checks FAILED -- review before running model.")


### Stage 9b - Basin-wide comparison table
Mirrors   logic. Populates the 
column in  for comparison against prior base years.

In [ ]:
# TAZ-level aggregation (mirrors sdfParcel_taz in notebook.ipynb)
# People_raw = people without the seasonal-adjustment factor
sdfParcel["_People_raw"] = sdfParcel["OccupiedUnits"] * (
    sdfParcel["PersonsPerUnit"].fillna(0) / HOUSEHOLD_SIZE_ADJUSTMENT
)
sdfParcel_taz = (
    sdfParcel.dropna(subset=["TAZ"])
    .groupby("TAZ", as_index=False)
    .agg(
        Residential_Units=("Residential_Units_Adjusted", "sum"),
        OccupiedUnits    =("OccupiedUnits",  "sum"),
        SeasonalUnits    =("SeasonalUnits",  "sum"),
        People           =("People",         "sum"),
        People_raw       =("_People_raw",    "sum"),
        LowUnits         =("LowUnits",       "sum"),
        MediumUnits      =("MediumUnits",    "sum"),
        HighUnits        =("HighUnits",      "sum"),
    )
)
sdfParcel.drop(columns=["_People_raw"], inplace=True)

# Round at TAZ level (sum floats first, then round -- preserves partial people)
for col in ["OccupiedUnits", "HighUnits", "MediumUnits", "LowUnits"]:
    sdfParcel_taz[col] = sdfParcel_taz[col].round(0).astype(int)

# Income tie-break: if TAZ has occupied units but all income classes rounded to 0
income_sum_taz = sdfParcel_taz["HighUnits"] + sdfParcel_taz["MediumUnits"] + sdfParcel_taz["LowUnits"]
fix_income_taz = (sdfParcel_taz["OccupiedUnits"] > 0) & (income_sum_taz == 0)
if fix_income_taz.sum() > 0:
    print(f"  Income fix applied to {fix_income_taz.sum()} TAZs")
    best_class = sdfParcel_taz.loc[fix_income_taz, ["HighUnits", "MediumUnits", "LowUnits"]].idxmax(axis=1)
    for idx, cls in best_class.items():
        sdfParcel_taz.loc[idx, cls] = 1
else:
    print("  Income fix: no TAZs affected")

sdfParcel_taz["People"] = sdfParcel_taz["People"].round(0).astype(int)

# Zero out People where OccupiedUnits == 0 at TAZ level
pop_fix_taz = (sdfParcel_taz["People"] > 0) & (sdfParcel_taz["OccupiedUnits"] == 0)
if pop_fix_taz.sum() > 0:
    print(f"  TAZ population zeroed for {pop_fix_taz.sum()} TAZs with 0 occupied units")
    sdfParcel_taz.loc[pop_fix_taz, "People"] = 0
sdfParcel_taz.to_csv(OUT_DIR / f"taz_summarized.csv", index=False)

# Prepare value dict for basin summary csv
value_dict = {
    "lodging occupancy rate"   : sdfParcel.loc[sdfParcel["TAU_TYPE"].isin(["HotelMotel", "Resort", "Casino"]), "TAU_Occupancy_Rate"].mean(),
    "campground occupancy rate": df_camp["Occupancy_Rate"].mean(),
    "house(VHR) rate"          : sdfParcel.loc[sdfParcel["VHR"] == "Yes", "VHR_Occupancy_Rate"].mean(),
    "seasonal rate"            : 0.39,
    "lodging unit"             : sdfParcel["TouristAccommodation_Units"].sum(),
    "campground"               : df_camp["Total_Sites"].sum(),
    "percentHouseSeasonal"     : sdfParcel_taz["SeasonalUnits"].sum() / (sdfParcel_taz["Residential_Units"].sum() - sdfParcel_taz["OccupiedUnits"].sum()),
    "school enrollment"        : (df_school[["elementary_school_enrollment", "middle_school_enrollment",
                                              "high_school_enrollment", "college_enrollment"]].sum(axis=1)).sum(),
    "employment"               : 26777,  # placeholder -- update with current employment data
    "residential unit"         : sdfParcel["Residential_Units"].sum(),
    "total persons"            : sdfParcel_taz["People"].sum(),
    "total persons raw"        : sdfParcel_taz["People_raw"].sum(),
    "census occupancy rate"    : (sdfParcel_taz["OccupiedUnits"].sum() / sdfParcel_taz["Residential_Units"].sum()),
    "low income res unit"      : sdfParcel_taz["LowUnits"].sum(),
    "medium income res unit"   : sdfParcel_taz["MediumUnits"].sum(),
    "high income res unit"     : sdfParcel_taz["HighUnits"].sum(),
    "total occupied unit"      : sdfParcel_taz["OccupiedUnits"].sum(),
    "persons per occupied unit": (sdfParcel_taz["People"].sum() / sdfParcel_taz["OccupiedUnits"].sum()),
}

# Map values into the prior-year comparison table
inputs_copy = OUT_DIR / "inputs_summarized copy.csv"
if inputs_copy.exists():
    dfInputs = pd.read_csv(inputs_copy)
    dfInputs["Housing_base_year_2024"] = 0
    dfInputs["total persons raw"]     = 0
    dfInputs["Housing_base_year_2024"] = dfInputs["category"].map(value_dict)
    dfInputs = dfInputs.round(2)
    dfInputs.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)
    dfInputs.to_csv(OUT_DIR / f"inputs_summarized.csv", index=False)
    print(f"inputs_summarized_.csv updated")
    display(dfInputs)
    dfInputs.to_csv(OUT_DIR / f"inputs_summarized.csv", index=False)
else:
    print(f"Reference file not found: {inputs_copy}")
    print("Basin-wide summary values:")
    for k, v in value_dict.items():
        print(f"  {k:<30}: {v:.4f}" if isinstance(v, float) else f"  {k:<30}: {v}")
        # k,v to csv
    summary_df = pd.DataFrame(list(value_dict.items()), columns=["category", "value"])
    summary_df.to_csv(OUT_DIR / f"inputs_summarized.csv", index=False)
    print(f"Summary values saved to inputs_summarized.csv")



In [ ]:
print(sdfParcel["SeasonalUnits"].sum())
print(sdfParcel["Residential_Units_Adjusted"].sum())
print(sdfParcel["Residential_Units"].sum())
print(sdfParcel["OccupiedUnits"].sum())